In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Carregando base de dados...")
df = pd.read_csv("ZIKA_BR_2018_2026_UNIFICADO.csv", low_memory=False)
print(f"Base carregada! Total de registros: {len(df)}")

display(df.describe())


In [ ]:
print("Limpando e transformando dados...")

# 1. Datas
date_cols = ['DT_NOTIFIC', 'DT_SIN_PRI', 'DT_OBITO', 'DT_ENCERRA']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. Categóricas (Mapeando '9' para 'Não Informado')
raca_map = {1.0: 'Branca', 2.0: 'Preta', 3.0: 'Amarela', 4.0: 'Parda', 5.0: 'Indígena', 9.0: 'Não Informado'}
df['CS_RACA_STR'] = df['CS_RACA'].map(raca_map).fillna('Não Informado')

sexo_map = {'M': 'Masculino', 'F': 'Feminino', 'I': 'Não Informado'}
df['CS_SEXO_STR'] = df['CS_SEXO'].map(sexo_map).fillna('Não Informado')

gestant_map = {1.0: '1º trim', 2.0: '2º trim', 3.0: '3º trim', 4.0: 'Ignorada', 5.0: 'Não', 6.0: 'Não se aplica', 9.0: 'Não Informado'}
df['CS_GESTANT_STR'] = df['CS_GESTANT'].map(gestant_map).fillna('Não Informado')

evolucao_map = {0.0: 'Em investigação', 1.0: 'Cura', 2.0: 'Óbito pelo agravo', 3.0: 'Óbito outra causa', 9.0: 'Não Informado'}
df['EVOLUCAO_STR'] = df['EVOLUCAO'].map(evolucao_map).fillna('Não Informado')

classi_map = {0.0: 'Descartado', 1.0: 'Confirmado', 2.0: 'Em investigação', 8.0: 'Inconclusivo'}
df['CLASSI_FIN_STR'] = df['CLASSI_FIN'].map(classi_map).fillna('Não Informado')

# 3. Decodificação de Idade
def decode_age(age_code):
    if pd.isna(age_code):
        return np.nan
    try:
        age_str = str(int(age_code)).zfill(4)
        unit = int(age_str[0])
        value = int(age_str[1:])
        if unit == 4: return value # Anos
        elif unit == 3: return value / 12.0 # Meses em anos
        elif unit == 2: return value / 365.25 # Dias em anos
        elif unit == 1: return value / (365.25 * 24) # Horas em anos
        else: return np.nan
    except:
        return np.nan

df['IDADE_ANOS'] = df['NU_IDADE_N'].apply(decode_age)

print("Transformação concluída!")
df[['DT_SIN_PRI', 'CS_RACA_STR', 'IDADE_ANOS', 'CS_GESTANT_STR']].head()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import pandas as pd

# --- MAPEAMENTO DE CÓDIGO IBGE → SIGLA ---
ibge_para_uf = {
    11:'RO', 12:'AC', 13:'AM', 14:'RR', 15:'PA', 16:'AP', 17:'TO',
    21:'MA', 22:'PI', 23:'CE', 24:'RN', 25:'PB', 26:'PE', 27:'AL',
    28:'SE', 29:'BA', 31:'MG', 32:'ES', 33:'RJ', 35:'SP',
    41:'PR', 42:'SC', 43:'RS',
    50:'MS', 51:'MT', 52:'GO', 53:'DF'
}

df['ANO_MES_DT'] = df['DT_SIN_PRI'].dt.to_period('M').dt.to_timestamp()
df = df[df['ANO_MES_DT'] >= '2018-01-01']

df['UF_SIGLA'] = df['SG_UF_NOT'].map(ibge_para_uf).fillna(df['SG_UF_NOT'].astype(str))

top_5_ufs = df['UF_SIGLA'].value_counts().nlargest(5).index.tolist()

df['UF_AGRUPADA'] = df['UF_SIGLA'].apply(lambda x: x if x in top_5_ufs else 'Outros Estados')

uf_ts = df.groupby(['ANO_MES_DT', 'UF_AGRUPADA']).size().unstack(fill_value=0)

cols = ['Outros Estados'] + top_5_ufs
cols = [c for c in cols if c in uf_ts.columns]
uf_ts = uf_ts[cols]

uf_ts_smooth = uf_ts.rolling(window=3, min_periods=1).mean()

cores = ['#C8C5BC', '#1D9E75', '#378ADD', '#D4537E', '#EF9F27', '#E24B4A'][:len(cols)]

fig, ax = plt.subplots(figsize=(20, 8), dpi=120)

uf_ts_smooth.plot(
    kind='area', stacked=True, ax=ax,
    color=cores, alpha=0.82, linewidth=1.0
)

ax.set_title('Curva Epidêmica de Zika (2018–2026): Composição por UF',
             fontsize=16, fontweight='bold', pad=14)
ax.set_xlabel('Período dos Primeiros Sintomas', fontsize=12, labelpad=8)
ax.set_ylabel('Casos Notificados (suavizados)', fontsize=12, labelpad=8)

# --- EIXO X COM ANOS GARANTIDOS ---
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_xlim(
    pd.Timestamp('2018-01-01'),
    df['ANO_MES_DT'].max() + pd.DateOffset(months=1)
)

for label in ax.get_xticklabels():
    label.set_visible(True)
    label.set_fontsize(11)
    label.set_rotation(0)
    label.set_ha('center')

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{int(x)}')
)
ax.tick_params(axis='y', labelsize=11)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          title='UF', title_fontsize=11, fontsize=10,
          loc='upper left', frameon=True, framealpha=0.9, edgecolor='#cccccc')

ax.grid(axis='y', linestyle='--', alpha=0.35, color='gray')
ax.grid(axis='x', visible=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_alpha(0.3)
ax.spines['bottom'].set_alpha(0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de barras empilhadas por UF e Classificação Final
uf_class_counts = df.groupby(['SG_UF_NOT', 'CLASSI_FIN_STR']).size().unstack(fill_value=0)

# Ordenar pelas UFs com mais casos gerais
uf_class_counts['TOTAL'] = uf_class_counts.sum(axis=1)
uf_class_counts = uf_class_counts.sort_values('TOTAL', ascending=False).drop('TOTAL', axis=1)

uf_class_counts.plot(kind='bar', stacked=True, figsize=(16, 7), colormap='Set2')
plt.title('Casos por Unidade Federativa e Situação (Classificação Final)', fontsize=16)
plt.xlabel('Unidade Federativa (UF)')
plt.ylabel('Total de Notificações')
plt.legend(title='Classificação Final')
plt.xticks(rotation=0)
plt.show()


In [ ]:
# Filtrar apenas mulheres para analisar a variável Gestante
mulheres = df[df['CS_SEXO'] == 'F']

# Excluir 'Não se aplica' (já que estamos olhando só mulheres)
gestantes = mulheres[mulheres['CS_GESTANT_STR'].isin(['1º trim', '2º trim', '3º trim', 'Ignorada', 'Não', 'Não Informado'])]

plt.figure(figsize=(14, 6))
sns.countplot(data=gestantes, x='CS_GESTANT_STR', 
              order=['1º trim', '2º trim', '3º trim', 'Ignorada', 'Não', 'Não Informado'],
              palette='coolwarm')
plt.title('Situação Gestacional das Mulheres Notificadas com Zika', fontsize=16)
plt.xlabel('Trimestre de Gestação (ou Não Gestante)')
plt.ylabel('Número de Notificações')
plt.show()

print("\n--- ATENÇÃO EPIDEMIOLÓGICA ---")
total_gestantes = len(gestantes[gestantes['CS_GESTANT_STR'].isin(['1º trim', '2º trim', '3º trim'])])
print(f"Total de Gestantes Notificadas: {total_gestantes} (Requerem acompanhamento para Síndrome Congênita)")


In [ ]:
# Calcular a diferença em dias entre Sintoma e Notificação
df['LATENCIA_DIAS'] = (df['DT_NOTIFIC'] - df['DT_SIN_PRI']).dt.days

# Remover valores negativos errados ou muito extremos (outliers)
latencia_valida = df[(df['LATENCIA_DIAS'] >= 0) & (df['LATENCIA_DIAS'] <= 100)]

plt.figure(figsize=(16, 6))
sns.histplot(latencia_valida['LATENCIA_DIAS'].dropna(), bins=50, kde=True, color='purple')
plt.title('Latência de Notificação (Dias entre 1º Sintoma e Notificação no SINAN)', fontsize=16)
plt.xlabel('Dias de Atraso (Latência)')
plt.ylabel('Frequência')
plt.axvline(latencia_valida['LATENCIA_DIAS'].median(), color='red', linestyle='--', label=f"Mediana: {latencia_valida['LATENCIA_DIAS'].median()} dias")
plt.legend()
plt.show()


In [ ]:
# Boxplot de idade por UF
plt.figure(figsize=(16, 7))
sns.boxplot(data=df, x='SG_UF_NOT', y='IDADE_ANOS', 
            order=df['SG_UF_NOT'].value_counts().index, 
            palette='pastel')

plt.title('Distribuição de Idade dos Notificados por UF', fontsize=16)
plt.xlabel('Unidade Federativa (UF) - Ordenada por Volume')
plt.ylabel('Idade (Anos)')
plt.show()


In [ ]:
# Taxa de campos "Não Informado" para Raça/Cor por UF
raca_ninf = df[df['CS_RACA_STR'] == 'Não Informado'].groupby('SG_UF_NOT').size()
total_uf = df.groupby('SG_UF_NOT').size()

taxa_ninf = (raca_ninf / total_uf * 100).fillna(0).sort_values(ascending=False)

plt.figure(figsize=(16, 6))
sns.barplot(x=taxa_ninf.index, y=taxa_ninf.values, palette='Reds_r')
plt.title('% de Fichas com Raça/Cor "Não Informada" por UF', fontsize=16)
plt.xlabel('UF')
plt.ylabel('Porcentagem (%)')
plt.axhline(taxa_ninf.mean(), color='red', linestyle='--', label=f'Média Nacional: {taxa_ninf.mean():.1f}%')
plt.legend()
plt.show()


In [ ]:
# Ver as informações gerais das colunas (quais são números, quais são textos)
df.info()

# Ver estatísticas básicas das colunas numéricas (média, mínimo, máximo)
df.describe()

# Ver quantos casos estão marcados como duplicados no sistema
df['NDUPLIC_N'].value_counts()
